In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
! # Make sure to enable the ipycanvas/ other widgets
!jupyter labextension enable widgetsnbextension

In [ ]:
import torch
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.sparse import csr_matrix
from matplotlib import pyplot as plt
from pathlib import Path
import anndata as ad
import os


from popari.model import Popari
from popari import pl, tl
from popari.util import concatenate
from popari.train import Trainer, BatchBlendTrainer, TrainParameters
from popari.components import PopariDataset
from popari.io import save_anndata, load_anndata

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection


In [ ]:
directory = "lloki_data/input/"

all_files = []
for root, dirs, files in os.walk(directory):
    for file in files:
        file_path = os.path.join(root, file)
        all_files.append(file_path)

print(all_files)

In [ ]:
import itertools
adata_dict = {}
for file_path in all_files:
    file_name = Path(file_path).stem
    adata = ad.read_h5ad(file_path)
    adata_dict[file_name] = adata


best_combo = None
max_overlap = 0
overlap_results = {}

for combo in itertools.combinations(adata_dict.keys(), 3):
    gene_sets = [set(adata_dict[dataset].var_names) for dataset in combo]
    overlap = gene_sets[0].intersection(gene_sets[1]).intersection(gene_sets[2])
    overlap_size = len(overlap)
    overlap_results[combo] = overlap_size
    
    if overlap_size > max_overlap:
        max_overlap = overlap_size
        best_combo = combo
        best_gene_set = overlap

print(f"Best combination: {best_combo}")
print(f"Number of overlapping genes: {max_overlap}")

datasets = {name: adata_dict[name] for name in best_combo}
gene_set = best_gene_set

In [ ]:
processed_datasets = []
dataset_names = list(datasets.keys())

for name, dataset in datasets.items():
    common_genes_indices = [i for i, gene in enumerate(dataset.var_names) if gene in gene_set]
    dataset_subset = dataset[:, common_genes_indices].copy()
    print(f"Subset {name}: {dataset_subset.n_obs} cells × {dataset_subset.n_vars} genes")
    processed_datasets.append(dataset_subset)

merged_dataset = ad.concat(
    processed_datasets, 
    label="batch", 
    keys=dataset_names, 
    merge="unique",
    uns_merge="unique"
)
merged_dataset

In [ ]:
sc.pp.normalize_total(merged_dataset, inplace=True)
sc.pp.log1p(merged_dataset)
sc.pp.highly_variable_genes(merged_dataset, n_top_genes=500)

In [ ]:
transformed_datasets = []
for i, name in enumerate(dataset_names):
    dataset_mask = merged_dataset.obs['batch'] == name
    dataset = merged_dataset[dataset_mask].copy()
    
    sc.pp.normalize_total(dataset, inplace=True)
    sc.pp.log1p(dataset)
    
    dataset = dataset[:, merged_dataset.var.highly_variable]
    
    transformed_datasets.append(PopariDataset(dataset, name))
    print(f"Transformed {name}: {dataset.n_obs} cells × {dataset.n_vars} genes")

In [ ]:
for transformed_dataset in transformed_datasets:
    transformed_dataset.compute_spatial_neighbors()

In [ ]:
save_anndata("lloki_data/preprocessed_dataset.h5ad", transformed_datasets)

In [ ]:
lambda_Sigma_x_inv=1e-4
lambda_Sigma_bar=1e-4
torch_context={
    "dtype": torch.float64,
    "device": "cuda:0"
}
K = 11
seed = 42

file = "lloki_data/preprocessed_dataset.h5ad"

original_adata = ad.read_h5ad("lloki_data/preprocessed_dataset.h5ad")

nmf_preiterations = 10
num_iterations = 50
model = Popari(                                                              
    K=K,                                                                        
    dataset_path=file,                                                                        
    lambda_Sigma_x_inv=lambda_Sigma_x_inv,
    spatial_affinity_mode="differential lookup",
    prior_x_modes = ["exponential shared fixed"]*3, 
    initial_context=torch_context,                                              
    torch_context=torch_context,
    initialization_method="leiden",
    hierarchical_levels=1,
    verbose=1,                                                                  
    random_state=seed,
)                  

train_parameters = TrainParameters(
    nmf_iterations=nmf_preiterations,
    iterations=num_iterations,
    savepath=(f"/home/raehashs/batch_effect_correction/popari/simulated_batch_data/output/trained_{num_iterations}_iterations.h5ad"),
)

trainer = Trainer(
    parameters=train_parameters,
    model=model,
    verbose=True,
)

trainer.train()

merged_dataset = concatenate(model.datasets)
original_adata.obsm["X_popari"] = merged_dataset.obsm["X"]





nmf_preiterations = 50
num_iterations = 0
model_2 = Popari(                                                              
    K=K,                                                                        
    dataset_path=file,                                                                        
    lambda_Sigma_x_inv=lambda_Sigma_x_inv,
    spatial_affinity_mode="differential lookup",
    initial_context=torch_context,                                              
    torch_context=torch_context,
    embedding_acceleration_trick=False,
    initialization_method="leiden",
    hierarchical_levels=1,
    prior_x_modes = ["cross_dataset_average"]*3, 
    verbose=1,                                                                  
    random_state=seed,
    batch_effect_correction="joint_metagenes",
)      


train_parameters_2 = TrainParameters(
    nmf_iterations=nmf_preiterations,
    iterations=num_iterations,
    savepath=(f"/home/raehashs/batch_effect_correction/popari/simulated_batch_data/output/trained_{num_iterations}_iterations_batch.h5ad"),
)

trainer_2 = BatchBlendTrainer(
    parameters=train_parameters_2,
    model=model_2,
    verbose=1,
)

trainer_2.train()  

for dataset in model_2.datasets:
    batch_effect_key = list(dataset.uns['batch_effect'].keys())[0]
    batch_effect = dataset.uns['batch_effect'][batch_effect_key]
    original_data = dataset.obsm['X']
    batch_corrected_data = original_data + batch_effect
    dataset.obsm['X_added_batch_blend'] = batch_corrected_data

merged_dataset_2 = concatenate(model_2.datasets)
original_adata.obsm["X_batchblend"] = merged_dataset_2.obsm["X"]
original_adata.obsm["X_added_batch_blend"] = merged_dataset_2.obsm["X_added_batch_blend"]


original_adata.write("lloki_data/analyzed_dataset.h5ad")


In [ ]:
original_adata = ad.read_h5ad("lloki_data/analyzed_dataset.h5ad")

In [ ]:
print(original_adata)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
sc.pp.neighbors(original_adata, use_rep="X_popari")
sc.tl.umap(original_adata)
sc.pl.umap(original_adata, color="high_level_annotation", alpha=0.5, ax=ax1, show=False, title="Cell types (Popari)")
sc.pl.umap(original_adata, color="batch", alpha=0.5, ax=ax2, show=False, title="Batches (Popari)")
plt.tight_layout()
fig.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
sc.pp.neighbors(original_adata, use_rep="X_batchblend")
sc.tl.umap(original_adata)
sc.pl.umap(original_adata, color="high_level_annotation", alpha=0.5, ax=ax1, show=False, title="Cell types (BatchBlend)")
sc.pl.umap(original_adata, color="batch", alpha=0.5, ax=ax2, show=False, title="Batches (BatchBlend)")
plt.tight_layout()
fig.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
sc.pp.neighbors(original_adata, use_rep="X_added_batch_blend")
sc.tl.umap(original_adata)
sc.pl.umap(original_adata, color="high_level_annotation", alpha=0.5, ax=ax1, show=False, title="Cell types (BatchBlend)")
sc.pl.umap(original_adata, color="batch", alpha=0.5, ax=ax2, show=False, title="Batches (BatchBlend)")
plt.tight_layout()
fig.show()

In [ ]:
bm = Benchmarker(
    original_adata,
    batch_key="batch",
    label_key="high_level_annotation",
    bio_conservation_metrics=BioConservation(),
    batch_correction_metrics=BatchCorrection(),
    embedding_obsm_keys=["X_popari", "X_batchblend"],
    n_jobs=1,
)
bm.benchmark()
bm.plot_results_table()
bm.plot_results_table(min_max_scale=False)

In [ ]:
df = bm.get_results(min_max_scale=False)
df.to_csv("lloki_data/analysis_42.csv")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
sc.pp.neighbors(original_adata, use_rep="X_umap")
sc.tl.umap(original_adata)
sc.pl.umap(original_adata, color="high_level_annotation", alpha=0.5, ax=ax1, show=False, title="Cell types (raw)")
sc.pl.umap(original_adata, color="batch", alpha=0.5, ax=ax2, show=False, title="Batches (raw)")
plt.tight_layout()
fig.show()

In [ ]:

for seed in [13, 28]:

    lambda_Sigma_x_inv=1e-4
    lambda_Sigma_bar=1e-4
    torch_context={
        "dtype": torch.float64,
        "device": "cuda:0"
    }
    K = 11
    
    file = "lloki_data/preprocessed_dataset.h5ad"
    
    original_adata = ad.read_h5ad("lloki_data/preprocessed_dataset.h5ad")
    
    nmf_preiterations = 10
    num_iterations = 50
    model = Popari(                                                              
        K=K,                                                                        
        dataset_path=file,                                                                        
        lambda_Sigma_x_inv=lambda_Sigma_x_inv,
        spatial_affinity_mode="differential lookup",
        prior_x_modes = ["exponential shared fixed"]*3, 
        initial_context=torch_context,                                              
        torch_context=torch_context,
        initialization_method="leiden",
        hierarchical_levels=1,
        verbose=1,                                                                  
        random_state=seed,
    )                  
    
    train_parameters = TrainParameters(
        nmf_iterations=nmf_preiterations,
        iterations=num_iterations,
        savepath=(f"/home/raehashs/batch_effect_correction/popari/simulated_batch_data/output/trained_{num_iterations}_iterations.h5ad"),
    )
    
    trainer = Trainer(
        parameters=train_parameters,
        model=model,
        verbose=True,
    )
    
    trainer.train()
    
    merged_dataset = concatenate(model.datasets)
    original_adata.obsm["X_popari"] = merged_dataset.obsm["X"]
    
    
    
    
    
    nmf_preiterations = 50
    num_iterations = 0
    model_2 = Popari(                                                              
        K=K,                                                                        
        dataset_path=file,                                                                        
        lambda_Sigma_x_inv=lambda_Sigma_x_inv,
        spatial_affinity_mode="differential lookup",
        initial_context=torch_context,                                              
        torch_context=torch_context,
        embedding_acceleration_trick=False,
        initialization_method="leiden",
        hierarchical_levels=1,
        prior_x_modes = ["cross_dataset_average"]*3, 
        verbose=1,                                                                  
        random_state=seed,
        batch_effect_correction="joint_metagenes",
    )      
    
    
    train_parameters_2 = TrainParameters(
        nmf_iterations=nmf_preiterations,
        iterations=num_iterations,
        savepath=(f"/home/raehashs/batch_effect_correction/popari/simulated_batch_data/output/trained_{num_iterations}_iterations_batch.h5ad"),
    )
    
    trainer_2 = BatchBlendTrainer(
        parameters=train_parameters_2,
        model=model_2,
        verbose=1,
    )
    
    trainer_2.train()  
    
    for dataset in model_2.datasets:
        batch_effect_key = list(dataset.uns['batch_effect'].keys())[0]
        batch_effect = dataset.uns['batch_effect'][batch_effect_key]
        original_data = dataset.obsm['X']
        batch_corrected_data = original_data + batch_effect
        dataset.obsm['X_added_batch_blend'] = batch_corrected_data
    
    merged_dataset_2 = concatenate(model_2.datasets)
    original_adata.obsm["X_batchblend"] = merged_dataset_2.obsm["X"]
    original_adata.obsm["X_added_batch_blend"] = merged_dataset_2.obsm["X_added_batch_blend"]
    
    
    original_adata.write(f"lloki_data/analyzed_dataset_{seed}.h5ad")


In [ ]:
for seed in ["13", "28", "42"]:
    print(seed)
    original_adata = ad.read_h5ad(f"lloki_data/analyzed_dataset_{seed}.h5ad")

    # fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    # sc.pp.neighbors(original_adata, use_rep="X_umap")
    # sc.tl.umap(original_adata)
    # sc.pl.umap(original_adata, color="high_level_annotation", alpha=0.5, ax=ax1, show=False, title="Cell types (raw)")
    # sc.pl.umap(original_adata, color="batch", alpha=0.5, ax=ax2, show=False, title="Batches (raw)")
    # plt.tight_layout()
    # fig.show()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    sc.pp.neighbors(original_adata, use_rep="X_popari")
    sc.tl.umap(original_adata)
    sc.pl.umap(original_adata, color="high_level_annotation", alpha=0.5, ax=ax1, show=False, title="Cell types (Popari)")
    sc.pl.umap(original_adata, color="batch", alpha=0.5, ax=ax2, show=False, title="Batches (Popari)")
    plt.tight_layout()
    fig.show()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    sc.pp.neighbors(original_adata, use_rep="X_batchblend")
    sc.tl.umap(original_adata)
    sc.pl.umap(original_adata, color="high_level_annotation", alpha=0.5, ax=ax1, show=False, title="Cell types (BatchBlend)")
    sc.pl.umap(original_adata, color="batch", alpha=0.5, ax=ax2, show=False, title="Batches (BatchBlend)")
    plt.tight_layout()
    fig.show()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    sc.pp.neighbors(original_adata, use_rep="X_batchblend")
    sc.tl.umap(original_adata)
    sc.pl.umap(original_adata, color="high_level_annotation", alpha=0.5, ax=ax1, show=False, title="Cell types (BatchBlend)")
    sc.pl.umap(original_adata, color="batch", alpha=0.5, ax=ax2, show=False, title="Batches (BatchBlend)")
    plt.tight_layout()
    fig.show()

    bm = Benchmarker(
    original_adata,
        batch_key="batch",
        label_key="high_level_annotation",
        bio_conservation_metrics=BioConservation(),
        batch_correction_metrics=BatchCorrection(),
        embedding_obsm_keys=["X_popari", "X_batchblend"],
        n_jobs=1,
    )
    bm.benchmark()
    bm.plot_results_table()
    bm.plot_results_table(min_max_scale=False)

    df = bm.get_results(min_max_scale=False)
    df.to_csv(f"lloki_data/analysis_{seed}.csv")
    

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

file_paths = ['lloki_data/analysis_13.csv', 'lloki_data/analysis_28.csv', 'lloki_data/analysis_42.csv']
dataframes = []
for path in file_paths:
    dataframes += [pd.read_csv(path)]
combined_df = pd.concat(dataframes, ignore_index=True)
embedding_types = combined_df['Embedding'].unique()
results = {
    'Embedding': [],
    'Metric': [],
    'Mean': [],
    'Std': []
}
# Metrics to analyze
metrics = ['Batch correction', 'Bio conservation', 'Total']
# Calculate statistics for each embedding and metric
for embedding in embedding_types:
    for metric in metrics:
        # Filter data for current embedding
        embedding_data = combined_df[combined_df['Embedding'] == embedding][metric]
        
        embedding_data = pd.to_numeric(embedding_data, errors='coerce')
        
        # Calculate mean and standard deviation
        mean_val = embedding_data.mean()
        std_val = embedding_data.std()
        
        # Store results
        results['Embedding'].append(embedding)
        results['Metric'].append(metric)
        results['Mean'].append(mean_val)
        results['Std'].append(std_val)
# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Create the bar plot
fig, ax = plt.subplots(figsize=(10, 6))

# Get the colors from seaborn's muted palette
colors = sns.color_palette("muted", 5)

color_val = [colors[3], colors[4]]

# Set up positions for bars
x = np.arange(len(metrics))
width = 0.35
multiplier = 0

# Plot bars for each embedding type
for i, embedding in enumerate(embedding_types):
    if embedding == "Metric Type":
        break
    embedding_data = results_df[results_df['Embedding'] == embedding]
    means = embedding_data['Mean'].values
    stds = embedding_data['Std'].values
    
    offset = width * multiplier
    # Don't add label to avoid automatic legend
    rects = ax.bar(x + offset, means, width, yerr=stds, 
                   capsize=5, alpha=0.8, color=color_val[i])
    
    multiplier += 1

# Customize the plot
ax.set_ylabel('Value')
ax.set_xlabel('Metrics')
ax.set_title('Aggregate Scoring Performance of Popari and BatchBlend on Coronal Mice Slices')
ax.set_xticks(x + width / 2)
ax.set_xticklabels(metrics)
ax.grid(axis='y', alpha=0.3)



popari_patch = plt.Rectangle((0, 0), 1, 1, fc=color_val[0], alpha=0.8)
batchblend_patch = plt.Rectangle((0, 0), 1, 1, fc=color_val[1], alpha=0.8)

ax.legend([popari_patch, batchblend_patch], ['Popari', 'BatchBlend'], 
          loc='upper left', bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# Load the dataframes
file_paths = ['lloki_data/analysis_13.csv', 'lloki_data/analysis_28.csv', 'lloki_data/analysis_42.csv']
dataframes = []
for path in file_paths:
    dataframes += [pd.read_csv(path)]
df_combined = pd.concat(dataframes)

df_filtered = df_combined[df_combined['Embedding'].isin(['X_popari', 'X_batchblend'])]
metric_columns = df_filtered.columns.drop('Embedding')
df_filtered[metric_columns] = df_filtered[metric_columns].apply(pd.to_numeric, errors='coerce')
averaged_metrics = df_filtered.groupby('Embedding').mean()

print(averaged_metrics)

In [ ]:
for file in all_files:
    print(f"results for {file}")
    adata, popari, batch_blend  = models[file]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    sc.pp.neighbors(adata, use_rep="ground_truth_X")
    sc.tl.umap(adata)
    sc.pl.umap(adata, color="cell_type", alpha=0.5, ax=ax1, show=False, title="Cell types (Ground Truth)")
    sc.pl.umap(adata, color="batch", alpha=0.5, ax=ax2, show=False, title="Batches (Ground Truth)")
    plt.tight_layout()
    fig.show()

    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    sc.pp.neighbors(adata, use_rep="X_popari")
    sc.tl.umap(adata)
    sc.pl.umap(adata, color="cell_type", alpha=0.5, ax=ax1, show=False, title="Cell types (Popari)")
    sc.pl.umap(adata, color="batch", alpha=0.5, ax=ax2, show=False, title="Batches (Popari)")
    plt.tight_layout()
    fig.show()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    sc.pp.neighbors(adata, use_rep="X_batchblend")
    sc.tl.umap(adata)
    sc.pl.umap(adata, color="cell_type", alpha=0.5, ax=ax1, show=False, title="Cell types (BatchBlend)")
    sc.pl.umap(adata, color="batch", alpha=0.5, ax=ax2, show=False, title="Batches (BatchBlend)")
    plt.tight_layout()
    fig.show()

    np.savetxt(f"/home/raehashs/batch_effect_correction/popari/simulated_batch_data/output/popari_" + str(file.split("0_")[-1:]) + ".csv", adata.obsm['X_popari'], delimiter=',', fmt='%d')
    np.savetxt(f"/home/raehashs/batch_effect_correction/popari/simulated_batch_data/output/batchblend_" + str(file.split("0_")[-1:]) + ".csv", adata.obsm['X_batchblend'], delimiter=',', fmt='%d')

    for dataset in batch_blend.datasets:
        batch_effect_key = list(dataset.uns['batch_effect'].keys())[0]
        batch_effect = dataset.uns['batch_effect'][batch_effect_key]
        original_data = dataset.obsm['X']
        batch_corrected_data = original_data + batch_effect
        dataset.obsm['X_added_batch_blend'] = batch_corrected_data
    print(original_data, batch_effect, batch_corrected_data)
    
    merged_dataset = concatenate(batch_blend.datasets)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    sc.pp.neighbors(merged_dataset, use_rep="X_added_batch_blend")
    sc.tl.umap(merged_dataset)
    sc.pl.umap(merged_dataset, color="cell_type", alpha=0.5, ax=ax1, show=False, title="Cell types (Batch Blend)")
    sc.pl.umap(merged_dataset, color="batch", alpha=0.5, ax=ax2, show=False, title="Batches (Batch Blend)")
    plt.tight_layout()
    fig.show()


    np.savetxt(f"/home/raehashs/batch_effect_correction/popari/simulated_batch_data/output/batch_plus_x_" + str(file.split("0_")[-1:]) + ".csv", merged_dataset.obsm['X_added_batch_blend'], delimiter=',', fmt='%d')
    
    bm = Benchmarker(
        adata,
        batch_key="batch",
        label_key="cell_type",
        bio_conservation_metrics=BioConservation(),
        batch_correction_metrics=BatchCorrection(),
        embedding_obsm_keys=["X_popari", "X_batchblend"],
        n_jobs=1,
    )
    bm.benchmark()
    bm.plot_results_table()
    bm.plot_results_table(min_max_scale=False)
    
    df = bm.get_results(min_max_scale=False)
    df.to_csv("/home/raehashs/batch_effect_correction/popari/simulated_batch_data/output/data_" + str(file.split("0_")[-1:]) + ".csv")

    path = f"/home/raehashs/batch_effect_correction/popari/simulated_batch_data/output/data_" + str(file.split("0_")[-1:]) + ".h5ad"
    adata.write(path)

In [ ]:
from popari._dataset_utils import _plot_all_embeddings

total_metagenes = 11
fov = 1
size= 8
for val in models:
    adata, popari, batch_blend = models[val]

    fig, axes = plt.subplots(3, total_metagenes, sharex=True, sharey=True, tight_layout=True, dpi=300, figsize=(total_metagenes, 3))
    _plot_all_embeddings.__wrapped__(popari.hierarchy[0].datasets[fov], size=size, embedding_key="ground_truth_X", colorbar=False, fig=fig, ax=axes[0, :].flat, edgecolors='none')
    _plot_all_embeddings.__wrapped__(popari.hierarchy[0].datasets[fov], size=size, colorbar=False, fig=fig, ax=axes[1, :K].flat, edgecolors='none')
    _plot_all_embeddings.__wrapped__(batch_blend.hierarchy[0].datasets[fov], size=size, colorbar=False, fig=fig, ax=axes[2, :K].flat, edgecolors='none')

    break

In [ ]:
from popari.util import unconcatenate
data = ad.read_h5ad("simulated_batch_data/output/batch_effect_3_with_50_percent/processed_dataset_13.h5ad")
data

In [ ]:
print(data.obsm['X_added_batch_blend'].shape,
data.obsm['X_batchblend'].shape,
data.obsm['X_popari'].shape, data.obsm['ground_truth_X'].shape)